# NUS ST5212 Survival Analysis — Detailed Case Study Notebook
## From censoring and risk sets to Kaplan–Meier, Nelson–Aalen, Cox PH, AFT, diagnostics, time-varying effects and Bayesian prediction

**Case study:** Rossi recidivism data (`lifelines.datasets.load_rossi`)

The dataset follows **432 people released from Maryland state prisons in the 1970s for up to one year**. The survival endpoint is **time to re-arrest**. A subject who is not re-arrested before follow-up ends is **right-censored** rather than treated as having an event at week 52.

This notebook is designed as a graduate-level tutorial aligned with the core statistical ideas usually associated with **ST5212 Survival Analysis**. It deliberately alternates between:

1. **probability theory and derivations**,  
2. **manual implementations** that expose the algorithm,  
3. **production-style estimators** from `lifelines`,  
4. **diagnostics and interpretation**, and  
5. **small experiments** showing what changes when assumptions change.

> **Ethical/data note.** This is a historical teaching dataset. Some fields are binary coded demographic/social indicators. We will not infer their substantive meaning beyond what is required for the statistical exercise unless an original codebook is available. The focus is methodology, not normative interpretation.

## Learning objectives

By the end, you should be able to explain and implement:

- the relationship among $F(t)$, $S(t)$, $f(t)$, $h(t)$ and $H(t)$;
- right censoring and the censored likelihood;
- risk sets and why they are the central data structure of survival analysis;
- Kaplan–Meier estimation and Greenwood uncertainty;
- Nelson–Aalen cumulative-hazard estimation;
- log-rank testing and its observed-versus-expected logic;
- exponential and Weibull parametric survival models;
- Cox proportional-hazards regression and partial likelihood;
- Efron handling of tied event times;
- hazard ratios and survival predictions;
- proportional-hazards diagnostics with Schoenfeld residuals;
- martingale/deviance residual intuition;
- stratified Cox regression;
- time-varying Cox models;
- Weibull/log-normal/log-logistic AFT regression;
- penalized Cox regression;
- Bayesian posterior predictive survival under a simple exponential model;
- competing-risk ideas using an explicit pedagogical extension.

The notebook also uses reusable classes so the same workflow can be applied to other survival datasets.

## 0. Mathematical map of the subject

Let $T\ge 0$ denote event time.

$$
F(t)=P(T\le t)
$$

$$
S(t)=P(T>t)=1-F(t)
$$

For continuous $T$,

$$
f(t)=\frac{dF(t)}{dt}=-\frac{dS(t)}{dt}
$$

The instantaneous hazard is

$$
h(t)=\lim_{\Delta t\to 0}
\frac{P(t\le T<t+\Delta t\mid T\ge t)}{\Delta t}
=\frac{f(t)}{S(t)}.
$$

Therefore

$$
h(t)=-\frac{d}{dt}\log S(t).
$$

Define cumulative hazard

$$
H(t)=\int_0^t h(u)\,du.
$$

Then

$$
H(t)=-\log S(t),
\qquad
\boxed{S(t)=e^{-H(t)}}.
$$

This identity is the bridge connecting essentially every estimator in this notebook.

## 1. Environment and reproducibility

The first cell installs the packages used in the case study. If the environment already contains them, installation is quick/no-op.

The visualizations use **Bokeh**. The statistical modelling uses `lifelines`; `scipy` is used to expose the Cox partial-likelihood optimizer manually.

In [1]:
# Run once per environment. Comment out after packages are available.
%pip install -q "lifelines>=0.30" "bokeh>=3.4" "statsmodels>=0.14" "scipy>=1.11" "scikit-learn>=1.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence
import math
import warnings

import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.preprocessing import StandardScaler

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool, Span
from bokeh.plotting import figure

from lifelines import (
    KaplanMeierFitter,
    NelsonAalenFitter,
    CoxPHFitter,
    CoxTimeVaryingFitter,
    ExponentialFitter,
    WeibullFitter,
    WeibullAFTFitter,
    LogNormalAFTFitter,
    LogLogisticAFTFitter,
    AalenJohansenFitter,
)
from lifelines.datasets import load_rossi
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import concordance_index, to_episodic_format

warnings.filterwarnings("ignore", category=FutureWarning)
output_notebook()

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

Loading BokehJS ...

## 2. Reusable architecture

To keep the notebook from turning into repeated procedural code, we use three small components:

- **Repository pattern:** one object owns data loading and validation.
- **Factory pattern:** one plotting utility creates consistently configured Bokeh figures.
- **Estimator objects:** manual Kaplan–Meier, Nelson–Aalen and Cox partial-likelihood calculations expose the algorithms.

The aim is not software-design ceremony. It is to separate **data access**, **statistical computation**, and **presentation**, which makes experimentation safer.

In [34]:
@dataclass(frozen=True)
class SurvivalConfig:
    duration_col: str = "week"
    event_col: str = "arrest"
    treatment_col: str = "fin"
    binary_covariates: tuple[str, ...] = ("fin", "race", "wexp", "mar", "paro")
    numeric_covariates: tuple[str, ...] = ("age", "prio")

    @property
    def covariates(self) -> list[str]:
        return list(self.binary_covariates + self.numeric_covariates)


class RossiRepository:
    """Repository pattern: centralize loading and lightweight validation."""

    FALLBACK_URL = (
        "https://raw.githubusercontent.com/CamDavidsonPilon/lifelines/"
        "master/lifelines/datasets/rossi.csv"
    )

    @staticmethod
    def load() -> pd.DataFrame:
        try:
            frame = load_rossi().copy()
        except Exception:
            frame = pd.read_csv(RossiRepository.FALLBACK_URL)
        return frame

    @staticmethod
    def validate(df: pd.DataFrame, cfg: SurvivalConfig) -> None:
        required = {cfg.duration_col, cfg.event_col, *cfg.covariates}
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")
        if (df[cfg.duration_col] < 0).any():
            raise ValueError("Durations must be non-negative.")
        if not set(df[cfg.event_col].dropna().unique()).issubset({0, 1}):
            raise ValueError("Event indicator must be binary 0/1.")


class PlotFactory:
    """Small Bokeh factory with safe explicit legend positions (no `location='best'`)."""

    @staticmethod
    def line(
        x,
        ys: Mapping[str, Iterable[float]],
        title: str,
        x_label: str,
        y_label: str,
        width: int = 760,
        height: int = 380,
        step: bool = False,
    ):
        p = figure(
            title=title,
            width=width,
            height=height,
            x_axis_label=x_label,
            y_axis_label=y_label,
            tools="pan,wheel_zoom,box_zoom,reset,save",
        )
        for label, y in ys.items():
            if step:
                p.step(x=x, y=y, mode="after", line_width=2.5, legend_label=label)
            else:
                p.line(x=x, y=y, line_width=2.5, legend_label=label)
        if p.legend:
            p.legend.location = "top_right"
            p.legend.click_policy = "hide"
        return p

    @staticmethod
    def scatter(
        x,
        y,
        title: str,
        x_label: str,
        y_label: str,
        width: int = 760,
        height: int = 380,
        hover: Mapping[str, str] | None = None,
        source_data: pd.DataFrame | None = None,
    ):
        p = figure(
            title=title,
            width=width,
            height=height,
            x_axis_label=x_label,
            y_axis_label=y_label,
            tools="pan,wheel_zoom,box_zoom,reset,save",
        )
        if source_data is None:
            p.scatter(x=x, y=y, size=7, alpha=0.65)
        else:
            src = ColumnDataSource(source_data)
            p.scatter(x=x, y=y, source=src, size=7, alpha=0.65)
            if hover:
                p.add_tools(HoverTool(tooltips=list(hover.items())))
        return p

    @staticmethod
    def histogram(values, title: str, x_label: str, bins: int = 20, width=760, height=350):
        hist, edges = np.histogram(np.asarray(values), bins=bins)
        p = figure(title=title, width=width, height=height, x_axis_label=x_label, y_axis_label="Count")
        p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.65, line_alpha=0.3)
        return p


CFG = SurvivalConfig()
df = RossiRepository.load()
RossiRepository.validate(df, CFG)

df.head()

,week,arrest,fin,age,race,wexp,mar,paro,prio
0,20,1,0,27,1,0,0,1,3
1,17,1,0,18,1,0,0,1,8
2,25,1,0,19,0,1,0,1,13
3,52,0,1,23,1,1,1,1,1
4,52,0,0,19,0,1,0,1,3


## 3. Understand the observation mechanism before modelling

For each subject there is a true event time $T_i$ and censoring time $C_i$. We observe

$$
Y_i=\min(T_i,C_i)
$$

and

$$
\delta_i=I(T_i\le C_i).
$$

If $\delta_i=1$, the event time is known. If $\delta_i=0$, we only know

$$
T_i>Y_i.
$$

This distinction prevents a common error: **week 52 with `arrest=0` does not mean re-arrest occurred at week 52**. It means the subject remained event-free through the observed follow-up.

The standard likelihood contribution is

$$
L_i(\theta)=f(Y_i;\theta)^{\delta_i}S(Y_i;\theta)^{1-\delta_i}.
$$

Using $f(t)=h(t)S(t)$,

$$
L_i(\theta)=h(Y_i;\theta)^{\delta_i}S(Y_i;\theta).
$$

That compact expression is one reason hazard-based modelling is so natural for censored data.

In [35]:
summary = pd.DataFrame({
    "n_subjects": [len(df)],
    "events": [int(df[CFG.event_col].sum())],
    "censored": [int((1 - df[CFG.event_col]).sum())],
    "event_fraction": [df[CFG.event_col].mean()],
    "median_observed_week": [df[CFG.duration_col].median()],
    "max_followup": [df[CFG.duration_col].max()],
})
display(summary)

display(df.describe().T)
show(PlotFactory.histogram(df[CFG.duration_col], "Observed follow-up durations", "Week", bins=20))

,n_subjects,events,censored,event_fraction,median_observed_week,max_followup
0,432,114,318,0.2639,52.0000,52


,count,mean,std,min,25%,50%,75%,max
week,432.0000,45.8542,12.6623,1.0000,50.0000,52.0000,52.0000,52.0000
arrest,432.0000,0.2639,0.4413,0.0000,0.0000,0.0000,1.0000,1.0000
fin,432.0000,0.5000,0.5006,0.0000,0.0000,0.5000,1.0000,1.0000
age,432.0000,24.5972,6.1134,17.0000,20.0000,23.0000,27.0000,44.0000
race,432.0000,0.8773,0.3285,0.0000,1.0000,1.0000,1.0000,1.0000
wexp,432.0000,0.5718,0.4954,0.0000,0.0000,1.0000,1.0000,1.0000
mar,432.0000,0.1227,0.3285,0.0000,0.0000,0.0000,0.0000,1.0000
paro,432.0000,0.6181,0.4864,0.0000,0.0000,1.0000,1.0000,1.0000
prio,432.0000,2.9838,2.8961,0.0000,1.0000,2.0000,4.0000,18.0000


### Interpretation checklist

Before estimating any survival curve, ask:

1. **How much censoring is present?** Heavy censoring means the right tail contains less information.
2. **Where does censoring occur?** Administrative censoring at a common study end behaves differently from dropout.
3. **Is censoring plausibly non-informative conditional on measured covariates?** If subjects leave precisely because their latent event risk has changed, standard methods can be biased.
4. **Are there many tied event times?** Weekly recording creates ties; Cox estimation must handle them deliberately.

## 4. Risk sets — the unifying data structure

At an event time $t_j$, define the risk set

$$
R(t_j)=\{i:Y_i\ge t_j\}.
$$

Let

$$
n_j=|R(t_j)|
$$

and let $d_j$ be the number of events exactly at $t_j$.

Many classical methods can be summarized from the same risk-set table:

$$
\hat S_{KM}(t)=\prod_{t_j\le t}\left(1-\frac{d_j}{n_j}\right)
$$

$$
\hat H_{NA}(t)=\sum_{t_j\le t}\frac{d_j}{n_j}
$$

The log-rank test compares observed versus expected $d_j$ across groups, while Cox regression weights members of $R(t_j)$ by $e^{X_i^T\beta}$.

In [5]:
def build_event_table(data: pd.DataFrame, duration_col: str, event_col: str) -> pd.DataFrame:
    times = np.sort(data.loc[data[event_col] == 1, duration_col].unique())
    rows = []
    for t in times:
        at_risk = int((data[duration_col] >= t).sum())
        events = int(((data[duration_col] == t) & (data[event_col] == 1)).sum())
        censored = int(((data[duration_col] == t) & (data[event_col] == 0)).sum())
        rows.append({"time": t, "n_risk": at_risk, "d_events": events, "censored_at_t": censored})
    return pd.DataFrame(rows)

risk_table = build_event_table(df, CFG.duration_col, CFG.event_col)
display(risk_table.head(12))

,time,n_risk,d_events,censored_at_t
0,1,432,1,0
1,2,431,1,0
2,3,430,1,0
3,4,429,1,0
4,5,428,1,0
5,6,427,1,0
6,7,426,1,0
7,8,425,5,0
8,9,420,2,0
9,10,418,1,0


## 5. Kaplan–Meier from first principles

At event time $t_j$, estimate the conditional probability of surviving the event time by

$$
\widehat{P}(T>t_j\mid T\ge t_j)
=1-\frac{d_j}{n_j}.
$$

The chain rule of probability gives

$$
P(T>t_k)=\prod_{j\le k}P(T>t_j\mid T\ge t_j),
$$

which yields

$$
\boxed{
\hat S_{KM}(t)=
\prod_{t_j\le t}
\left(1-\frac{d_j}{n_j}\right)
}.
$$

A useful theorem-level fact is that, under the standard independent-censoring setup, Kaplan–Meier is the **nonparametric maximum-likelihood estimator** of the survival distribution.

Greenwood's variance estimate is

$$
\widehat{\mathrm{Var}}\{\hat S(t)\}
\approx
\hat S(t)^2
\sum_{t_j\le t}
\frac{d_j}{n_j(n_j-d_j)}.
$$

Notice how uncertainty tends to grow in the tail: $n_j$ becomes small.

In [6]:
class ManualKaplanMeier:
    def fit(self, durations: Sequence[float], events: Sequence[int]) -> "ManualKaplanMeier":
        x = pd.DataFrame({"time": durations, "event": events}).sort_values("time")
        event_times = np.sort(x.loc[x["event"] == 1, "time"].unique())

        survival = 1.0
        greenwood_sum = 0.0
        rows = []

        for t in event_times:
            n = int((x["time"] >= t).sum())
            d = int(((x["time"] == t) & (x["event"] == 1)).sum())
            survival *= (1.0 - d / n)
            if n - d > 0:
                greenwood_sum += d / (n * (n - d))
            variance = (survival ** 2) * greenwood_sum
            rows.append({
                "time": t,
                "n_risk": n,
                "d_events": d,
                "survival": survival,
                "greenwood_var": variance,
                "greenwood_se": math.sqrt(max(variance, 0.0)),
            })

        self.table_ = pd.DataFrame(rows)
        return self


manual_km = ManualKaplanMeier().fit(df[CFG.duration_col], df[CFG.event_col])
display(manual_km.table_.head(12))

,time,n_risk,d_events,survival,greenwood_var,greenwood_se
0,1,432,1,0.9977,0.0000,0.0023
1,2,431,1,0.9954,0.0000,0.0033
2,3,430,1,0.9931,0.0000,0.0040
3,4,429,1,0.9907,0.0000,0.0046
4,5,428,1,0.9884,0.0000,0.0051
5,6,427,1,0.9861,0.0000,0.0056
6,7,426,1,0.9838,0.0000,0.0061
7,8,425,5,0.9722,0.0001,0.0079
8,9,420,2,0.9676,0.0001,0.0085
9,10,418,1,0.9653,0.0001,0.0088


In [36]:
km = KaplanMeierFitter(label="Overall KM")
km.fit(df[CFG.duration_col], event_observed=df[CFG.event_col])

km_curve = km.survival_function_.reset_index()
km_curve.columns = ["time", "survival"]

manual_curve = pd.concat([
    pd.DataFrame({"time": [0], "survival": [1.0]}),
    manual_km.table_[["time", "survival"]]
], ignore_index=True)

p = figure(
    title="Kaplan–Meier: manual estimator vs lifelines",
    width=820,
    height=420,
    x_axis_label="Week",
    y_axis_label="Estimated survival P(T > t)",
)
p.step(manual_curve["time"], manual_curve["survival"], mode="after", line_width=3, legend_label="Manual KM")
p.step(km_curve["time"], km_curve["survival"], mode="after", line_width=2, line_dash="dashed", legend_label="lifelines KM")
p.legend.location = "bottom_left"
p.legend.click_policy = "hide"
show(p)

print(f"Median survival time: {km.median_survival_time_}")
print("If this is infinity, fewer than 50% of subjects experienced the event during follow-up.")

Median survival time: inf
If this is infinity, fewer than 50% of subjects experienced the event during follow-up.


### Why the median can be infinite

The Kaplan–Meier median is the first time $t$ for which

$$
\hat S(t)\le 0.5.
$$

If the curve never reaches $0.5$ during observed follow-up, the sample does **not identify the median event time**. Returning infinity is not a software failure; it is an honest statistical statement that more than half the cohort remained event-free within the observation window.

## 6. Nelson–Aalen cumulative hazard

Instead of multiplying survival decrements, Nelson–Aalen adds estimated hazard increments:

$$
\Delta\hat H(t_j)=\frac{d_j}{n_j}.
$$

Therefore

$$
\boxed{
\hat H_{NA}(t)=\sum_{t_j\le t}\frac{d_j}{n_j}
}.
$$

Since

$$
S(t)=e^{-H(t)},
$$

we obtain another survival estimate

$$
\hat S_{NA}(t)=e^{-\hat H_{NA}(t)}.
$$

Why is this close to Kaplan–Meier? For small $x$,

$$
\log(1-x)\approx -x.
$$

Hence

$$
\log\hat S_{KM}(t)
=
\sum_j\log\left(1-\frac{d_j}{n_j}\right)
\approx
-\sum_j\frac{d_j}{n_j}
=-\hat H_{NA}(t).
$$

In [8]:
class ManualNelsonAalen:
    def fit(self, durations, events) -> "ManualNelsonAalen":
        x = pd.DataFrame({"time": durations, "event": events})
        event_times = np.sort(x.loc[x["event"] == 1, "time"].unique())
        cumulative = 0.0
        rows = []
        for t in event_times:
            n = int((x["time"] >= t).sum())
            d = int(((x["time"] == t) & (x["event"] == 1)).sum())
            increment = d / n
            cumulative += increment
            rows.append({
                "time": t,
                "n_risk": n,
                "d_events": d,
                "hazard_increment": increment,
                "cum_hazard": cumulative,
                "survival_from_H": np.exp(-cumulative),
            })
        self.table_ = pd.DataFrame(rows)
        return self


na_manual = ManualNelsonAalen().fit(df[CFG.duration_col], df[CFG.event_col])
naf = NelsonAalenFitter(label="Nelson–Aalen")
naf.fit(df[CFG.duration_col], event_observed=df[CFG.event_col])

display(na_manual.table_.head(12))

,time,n_risk,d_events,hazard_increment,cum_hazard,survival_from_H
0,1,432,1,0.0023,0.0023,0.9977
1,2,431,1,0.0023,0.0046,0.9954
2,3,430,1,0.0023,0.0070,0.9931
3,4,429,1,0.0023,0.0093,0.9908
4,5,428,1,0.0023,0.0116,0.9884
5,6,427,1,0.0023,0.0140,0.9861
6,7,426,1,0.0023,0.0163,0.9838
7,8,425,5,0.0118,0.0281,0.9723
8,9,420,2,0.0048,0.0328,0.9677
9,10,418,1,0.0024,0.0352,0.9654


In [9]:
km_at_events = manual_km.table_.set_index("time")["survival"]
na_at_events = na_manual.table_.set_index("time")["survival_from_H"]
comparison = pd.concat([km_at_events, na_at_events], axis=1).reset_index()
comparison.columns = ["time", "KM", "exp(-Nelson_Aalen)"]
comparison["absolute_gap"] = (comparison["KM"] - comparison["exp(-Nelson_Aalen)"]).abs()

display(comparison.head(12))

show(PlotFactory.line(
    comparison["time"],
    {"Kaplan–Meier": comparison["KM"], "exp(-Nelson–Aalen)": comparison["exp(-Nelson_Aalen)"]},
    "Two nonparametric routes to survival",
    "Week",
    "Estimated survival",
    step=True,
))

,time,KM,exp(-Nelson_Aalen),absolute_gap
0,1,0.9977,0.9977,0.0000
1,2,0.9954,0.9954,0.0000
2,3,0.9931,0.9931,0.0000
3,4,0.9907,0.9908,0.0000
4,5,0.9884,0.9884,0.0000
5,6,0.9861,0.9861,0.0000
6,7,0.9838,0.9838,0.0000
7,8,0.9722,0.9723,0.0001
8,9,0.9676,0.9677,0.0001
9,10,0.9653,0.9654,0.0001


## 7. Treatment-group survival and the log-rank test

The variable `fin` represents assignment to an experimental financial-aid treatment in the Rossi data description.

A plot can suggest separation, but visual separation is not itself a hypothesis test.

For two groups, at event time $t_j$, let $n_{1j}$ be the number at risk in group 1, $n_j$ total at risk, and $d_j$ total events. Under equal hazards, expected group-1 events are

$$
E_{1j}=d_j\frac{n_{1j}}{n_j}.
$$

The log-rank score accumulates

$$
U=\sum_j(O_{1j}-E_{1j}).
$$

After variance standardization,

$$
Z=\frac{U}{\sqrt{V(U)}}\approx N(0,1),
$$

so

$$
Z^2\approx\chi^2_1.
$$

A deep connection: for a Cox model containing only one binary group covariate, the **Cox score test for $\beta=0$ is essentially the log-rank test**.

In [37]:
km_by_fin = {}
for value, group in df.groupby(CFG.treatment_col):
    fitter = KaplanMeierFitter(label=f"fin={value}")
    fitter.fit(group[CFG.duration_col], group[CFG.event_col])
    curve = fitter.survival_function_.reset_index()
    curve.columns = ["time", "survival"]
    km_by_fin[f"fin={value}"] = curve

p = figure(
    title="Kaplan–Meier curves by financial-aid assignment",
    width=820,
    height=420,
    x_axis_label="Week",
    y_axis_label="Estimated probability of remaining re-arrest free",
)
for label, curve in km_by_fin.items():
    p.step(curve["time"], curve["survival"], mode="after", line_width=3, legend_label=label)
p.legend.location = "bottom_left"
p.legend.click_policy = "hide"
show(p)

mask0 = df[CFG.treatment_col] == 0
mask1 = df[CFG.treatment_col] == 1

lr = logrank_test(
    df.loc[mask0, CFG.duration_col],
    df.loc[mask1, CFG.duration_col],
    event_observed_A=df.loc[mask0, CFG.event_col],
    event_observed_B=df.loc[mask1, CFG.event_col],
)

print(f"Log-rank statistic = {lr.test_statistic:.4f}")
print(f"p-value            = {lr.p_value:.6f}")

Log-rank statistic = 3.8376
p-value            = 0.050116


In [38]:
def manual_logrank(data: pd.DataFrame, duration_col: str, event_col: str, group_col: str, group1=1):
    event_times = np.sort(data.loc[data[event_col] == 1, duration_col].unique())
    U = 0.0
    V = 0.0
    rows = []

    for t in event_times:
        risk = data[data[duration_col] >= t]
        events_t = data[(data[duration_col] == t) & (data[event_col] == 1)]

        n = len(risk)
        n1 = int((risk[group_col] == group1).sum())
        d = len(events_t)
        o1 = int((events_t[group_col] == group1).sum())
        e1 = d * n1 / n

        if n > 1:
            v = (n1 * (n - n1) * d * (n - d)) / (n**2 * (n - 1))
        else:
            v = 0.0

        U += o1 - e1
        V += v
        rows.append({"time": t, "O1": o1, "E1": e1, "O_minus_E": o1-e1, "variance": v})

    z = U / np.sqrt(V)
    chi2 = z**2
    p = stats.chi2.sf(chi2, df=1)
    return pd.DataFrame(rows), {"U": U, "V": V, "z": z, "chi2": chi2, "p_value": p}


lr_table, lr_manual = manual_logrank(df, CFG.duration_col, CFG.event_col, CFG.treatment_col, group1=1)
display(lr_table.head(12))
display(pd.DataFrame([lr_manual]))

,time,O1,E1,O_minus_E,variance
0,1,0,0.5000,-0.5000,0.2500
1,2,0,0.5012,-0.5012,0.2500
2,3,0,0.5023,-0.5023,0.2500
3,4,0,0.5035,-0.5035,0.2500
4,5,0,0.5047,-0.5047,0.2500
5,6,0,0.5059,-0.5059,0.2500
6,7,1,0.5070,0.4930,0.2500
7,8,3,2.5294,0.4706,1.2380
8,9,2,1.0095,0.9905,0.4988
9,10,0,0.5024,-0.5024,0.2500


,U,V,z,chi2,p_value
0,-10.4256,28.3232,-1.9590,3.8376,0.0501


### Interpreting a log-rank result correctly

A small p-value supports evidence against equality of the two event-time processes under the assumptions of the test. It does **not** directly estimate effect size.

For effect size with covariate adjustment we move to Cox regression.

## 8. Parametric survival models: exponential and Weibull

### Exponential

$$
S(t)=e^{-\lambda t},
\qquad
h(t)=\lambda.
$$

The hazard is constant. Under right censoring, the exponential likelihood is

$$
L(\lambda)
=
\prod_i
\lambda^{\delta_i}e^{-\lambda t_i}
=
\lambda^D e^{-\lambda\sum_i t_i},
$$

where

$$
D=\sum_i\delta_i.
$$

Differentiate the log-likelihood:

$$
\ell(\lambda)=D\log\lambda-\lambda\sum_i t_i.
$$

Setting the score to zero gives a very intuitive MLE:

$$
\boxed{
\hat\lambda=\frac{D}{\sum_i t_i}
}
$$

= **number of events / total person-time at risk**.

### Weibull

Using the `lifelines` parameterization,

$$
S(t)=\exp\left[-\left(\frac{t}{\lambda}\right)^\rho\right]
$$

and

$$
h(t)=\frac{\rho}{\lambda}
\left(\frac{t}{\lambda}\right)^{\rho-1}.
$$

- $\rho=1$: constant hazard, exponential special case.
- $\rho>1$: increasing hazard.
- $0<\rho<1$: decreasing hazard.

In [39]:
D = df[CFG.event_col].sum()
person_time = df[CFG.duration_col].sum()
lambda_manual = D / person_time

expf = ExponentialFitter().fit(df[CFG.duration_col], df[CFG.event_col], label="Exponential")
wbf = WeibullFitter().fit(df[CFG.duration_col], df[CFG.event_col], label="Weibull")

param_summary = pd.DataFrame({
    "quantity": ["Manual exponential rate", "lifelines exponential lambda_", "Weibull lambda_", "Weibull rho_"],
    "value": [lambda_manual, expf.lambda_, wbf.lambda_, wbf.rho_],
})
display(param_summary)

print("Reminder: lifelines' ExponentialFitter lambda_ is a time-scale parameter, so rate = 1/lambda_.")
print(f"1 / fitted lambda_ = {1/expf.lambda_:.6f}")

,quantity,value
0,Manual exponential rate,0.0058
1,lifelines exponential lambda_,173.7632
2,Weibull lambda_,123.6771
3,Weibull rho_,1.3651


Reminder: lifelines' ExponentialFitter lambda_ is a time-scale parameter, so rate = 1/lambda_.
1 / fitted lambda_ = 0.005755


In [40]:
timeline = np.linspace(0.01, df[CFG.duration_col].max(), 200)
km_pred = km.predict(timeline).values
exp_pred = expf.survival_function_at_times(timeline).values
wei_pred = wbf.survival_function_at_times(timeline).values

show(PlotFactory.line(
    timeline,
    {
        "Kaplan–Meier": km_pred,
        "Exponential": exp_pred,
        "Weibull": wei_pred,
    },
    "Nonparametric survival vs parametric assumptions",
    "Week",
    "Survival",
))

### Model-thinking exercise

The comparison above answers a fundamental statistical question:

> How much structure are we willing to impose on the hazard?

- Kaplan–Meier imposes very little shape structure.
- Exponential compresses the entire process to one constant hazard.
- Weibull adds one shape parameter, allowing monotonically increasing or decreasing hazard.

If a simple parametric curve tracks KM closely, the model can provide smoother estimates and extrapolation. If it misses systematic features, extra parametric efficiency is bought with misspecification bias.

## 9. Cox proportional hazards model

The Cox model is

$$
\boxed{
h(t\mid X)=h_0(t)e^{X^T\beta}
}
$$

with unspecified baseline hazard $h_0(t)$.

For two subjects $a$ and $b$,

$$
\frac{h(t\mid X_a)}{h(t\mid X_b)}
=
\exp\{(X_a-X_b)^T\beta\}.
$$

The baseline cancels, and the ratio contains no $t$. This is the **proportional-hazards assumption**.

For a one-unit increase in $X_j$,

$$
HR_j=e^{\beta_j}.
$$

- $HR_j>1$: higher instantaneous event hazard.
- $HR_j<1$: lower instantaneous event hazard.
- $HR_j=1$: no multiplicative hazard difference.

Do not confuse a hazard ratio with a probability ratio or a survival-time ratio.

In [41]:
cph = CoxPHFitter()
cph.fit(df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], duration_col=CFG.duration_col, event_col=CFG.event_col)

display(cph.summary[[
    "coef", "exp(coef)", "se(coef)", "coef lower 95%", "coef upper 95%",
    "exp(coef) lower 95%", "exp(coef) upper 95%", "p"
]].round(4))

print(f"Concordance index: {cph.concordance_index_:.4f}")

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,p
covariate,,,,,,,,
fin,-0.3794,0.6843,0.1914,-0.7545,-0.0043,0.4702,0.9957,0.0474
race,0.3139,1.3688,0.3080,-0.2898,0.9176,0.7484,2.5032,0.3081
wexp,-0.1498,0.8609,0.2122,-0.5657,0.2662,0.5679,1.3049,0.4803
mar,-0.4337,0.6481,0.3819,-1.1822,0.3147,0.3066,1.3699,0.2561
paro,-0.0849,0.9186,0.1958,-0.4685,0.2988,0.6259,1.3482,0.6646
age,-0.0574,0.9442,0.0220,-0.1006,-0.0143,0.9043,0.9858,0.0090
prio,0.0915,1.0958,0.0286,0.0353,0.1476,1.0360,1.1591,0.0014


Concordance index: 0.6403


In [15]:
def interpretation_table(cph_model: CoxPHFitter) -> pd.DataFrame:
    s = cph_model.summary.copy()
    out = pd.DataFrame(index=s.index)
    out["beta"] = s["coef"]
    out["HR"] = s["exp(coef)"]
    out["95% CI lower"] = s["exp(coef) lower 95%"]
    out["95% CI upper"] = s["exp(coef) upper 95%"]
    out["p"] = s["p"]
    out["direction"] = np.where(out["HR"] > 1, "higher hazard", "lower hazard")
    out["approx_percent_hazard_change"] = (out["HR"] - 1) * 100
    return out.sort_values("p")

cox_interpretation = interpretation_table(cph)
display(cox_interpretation.round(4))

,beta,HR,95% CI lower,95% CI upper,p,direction,approx_percent_hazard_change
covariate,,,,,,,
prio,0.0915,1.0958,1.0360,1.1591,0.0014,higher hazard,9.5814
age,-0.0574,0.9442,0.9043,0.9858,0.0090,lower hazard,-5.5819
fin,-0.3794,0.6843,0.4702,0.9957,0.0474,lower hazard,-31.5743
mar,-0.4337,0.6481,0.3066,1.3699,0.2561,lower hazard,-35.1896
race,0.3139,1.3688,0.7484,2.5032,0.3081,higher hazard,36.8753
wexp,-0.1498,0.8609,0.5679,1.3049,0.4803,lower hazard,-13.9116
paro,-0.0849,0.9186,0.6259,1.3482,0.6646,lower hazard,-8.1369


### Interpretation discipline

Suppose a fitted coefficient is $\hat\beta=-0.40$.

Then

$$
e^{-0.40}\approx0.67.
$$

The appropriate statement is approximately:

> holding the other modelled covariates fixed, a one-unit increase in that predictor is associated with an instantaneous event hazard about 33% lower, assuming proportional hazards.

It is **not** automatically correct to say “33% lower probability of re-arrest” or “33% longer survival”.

## 10. Cox partial likelihood — implement the core algorithm

Suppose subject $i$ fails at event time $t_i$ and $R(t_i)$ is the risk set.

Conditional on **someone in the risk set failing at that instant**, the Cox model gives

$$
P(i\text{ fails}\mid \text{one failure in }R(t_i))
=
\frac{h_0(t_i)e^{X_i^T\beta}}
{\sum_{j\in R(t_i)}h_0(t_i)e^{X_j^T\beta}}
$$

so $h_0(t_i)$ cancels:

$$
\boxed{
\frac{e^{X_i^T\beta}}
{\sum_{j\in R(t_i)}e^{X_j^T\beta}}
}.
$$

With no ties, multiply these terms across failures.

Weekly data contain ties, so below we implement the **Efron approximation**. If $D_t$ is the set of $d_t$ tied deaths at time $t$, the Efron log-partial likelihood contribution is

$$
\sum_{i\in D_t}\eta_i
-
\sum_{\ell=0}^{d_t-1}
\log\left[
\sum_{j\in R_t}e^{\eta_j}
-
\frac{\ell}{d_t}
\sum_{i\in D_t}e^{\eta_i}
\right],
$$

where $\eta_i=X_i^T\beta$.

In [42]:
class EfronCoxPartialLikelihood:
    """Educational Cox optimizer using the Efron tied-event partial likelihood."""

    def __init__(self, maxiter: int = 1000):
        self.maxiter = maxiter

    @staticmethod
    def _neg_log_partial_likelihood(beta, X, time, event):
        eta = X @ beta
        # Clip only for numerical safety in this educational implementation.
        exp_eta = np.exp(np.clip(eta, -50, 50))
        ll = 0.0

        for t in np.sort(np.unique(time[event == 1])):
            deaths = (time == t) & (event == 1)
            risk = time >= t
            d = int(deaths.sum())

            ll += eta[deaths].sum()
            risk_sum = exp_eta[risk].sum()
            death_sum = exp_eta[deaths].sum()

            for ell in range(d):
                denom = risk_sum - (ell / d) * death_sum
                if denom <= 0:
                    return np.inf
                ll -= np.log(denom)

        return -ll

    def fit(self, X: pd.DataFrame, time: pd.Series, event: pd.Series):
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        result = optimize.minimize(
            self._neg_log_partial_likelihood,
            x0=np.zeros(X.shape[1]),
            args=(Xs, time.to_numpy(float), event.to_numpy(int)),
            method="BFGS",
            options={"maxiter": self.maxiter, "gtol": 1e-7},
        )

        self.scaler_ = scaler
        self.result_ = result
        self.coef_standardized_ = pd.Series(result.x, index=X.columns, name="manual_efron_beta")
        return self


X = df[CFG.covariates]
manual_cox = EfronCoxPartialLikelihood().fit(X, df[CFG.duration_col], df[CFG.event_col])

# Fit lifelines on exactly the same standardized design for an apples-to-apples comparison.
Xs = pd.DataFrame(manual_cox.scaler_.transform(X), columns=X.columns, index=X.index)
cox_std_df = pd.concat([df[[CFG.duration_col, CFG.event_col]], Xs], axis=1)
cph_std = CoxPHFitter().fit(cox_std_df, CFG.duration_col, CFG.event_col)

coef_compare = pd.concat([
    manual_cox.coef_standardized_,
    cph_std.params_.rename("lifelines_beta")
], axis=1)
coef_compare["absolute_gap"] = (coef_compare["manual_efron_beta"] - coef_compare["lifelines_beta"]).abs()

display(coef_compare.round(6))
print("Optimizer success:", manual_cox.result_.success)
print("Optimizer message:", manual_cox.result_.message)

,manual_efron_beta,lifelines_beta,absolute_gap
fin,-0.1897,-0.1897,0.0000
race,0.1030,0.1030,0.0000
wexp,-0.0741,-0.0741,0.0000
mar,-0.1423,-0.1423,0.0000
paro,-0.0412,-0.0412,0.0000
age,-0.3507,-0.3507,0.0000
prio,0.2647,0.2647,0.0000


Optimizer success: False
Optimizer message: Desired error not necessarily achieved due to precision loss.


### What the optimizer is doing

The score function for untied Cox data can be written

$$
U(\beta)
=
\sum_{i:\delta_i=1}
\left[
X_i-\bar X(\beta,t_i)
\right]
$$

where

$$
\bar X(\beta,t)
=
\frac{
\sum_{j\in R(t)}X_j e^{X_j^T\beta}
}{
\sum_{j\in R(t)}e^{X_j^T\beta}
}.
$$

So estimation repeatedly balances

$$
\text{covariates of actual failures}
-
\text{risk-weighted covariates expected to fail}.
$$

Newton/BFGS-type optimization changes $\beta$ until this discrepancy is approximately zero.

Under regularity conditions,

$$
\sqrt n(\hat\beta-\beta_0)
\xrightarrow{d}
N(0,I(\beta_0)^{-1}),
$$

which is the basis of coefficient standard errors, Wald tests and confidence intervals.

## 11. Baseline cumulative hazard and subject-specific survival curves

After $\hat\beta$ is known, Cox can estimate baseline cumulative hazard with a Breslow-type estimator:

$$
\hat H_0(t)
=
\sum_{t_j\le t}
\frac{d_j}
{\sum_{i\in R(t_j)}e^{X_i^T\hat\beta}}.
$$

Then

$$
\hat S_0(t)=e^{-\hat H_0(t)}.
$$

For covariates $X$,

$$
\boxed{
\hat S(t\mid X)
=
\left[\hat S_0(t)\right]^{e^{X^T\hat\beta}}
}.
$$

Thus a Cox model is not limited to hazard ratios; it can produce an entire predicted survival curve.

In [43]:
# Select representative low/median/high risk profiles using fitted partial hazard.
risk_score = cph.predict_partial_hazard(df[CFG.covariates]).rename("risk")
order = risk_score.sort_values()
idx_low = order.index[int(0.1 * (len(order)-1))]
idx_mid = order.index[int(0.5 * (len(order)-1))]
idx_high = order.index[int(0.9 * (len(order)-1))]
profiles = df.loc[[idx_low, idx_mid, idx_high], CFG.covariates].copy()
profiles.index = ["10th pct risk", "50th pct risk", "90th pct risk"]

display(profiles)

surv = cph.predict_survival_function(profiles)
p = figure(
    title="Cox-predicted survival for representative risk profiles",
    width=820,
    height=430,
    x_axis_label="Week",
    y_axis_label="Predicted P(T > t | X)",
)
for col in surv.columns:
    p.line(surv.index, surv[col], line_width=3, legend_label=str(col))
p.legend.location = "bottom_left"
p.legend.click_policy = "hide"
show(p)

,fin,race,wexp,mar,paro,age,prio
10th pct risk,0,1,1,0,0,38,0
50th pct risk,1,1,0,0,1,20,1
90th pct risk,0,1,1,0,1,19,5


## 12. Concordance index: discrimination, not calibration

For comparable subject pairs, the concordance index asks whether the subject predicted to have greater risk tends to experience the event earlier.

Very roughly,

$$
C=P(\text{predicted ordering agrees with observed ordering}).
$$

- $C\approx0.5$: near-random ranking.
- $C\to1$: strong ranking discrimination.

But C-index does **not** say the predicted survival probabilities are numerically calibrated. A model can rank subjects correctly while producing poorly calibrated absolute risks.

In [44]:
print(f"Training concordance index = {cph.concordance_index_:.4f}")

# A simple sensitivity check: compare full model with reduced covariate sets.
model_specs = {
    "treatment only": ["fin"],
    "age + prior": ["age", "prio"],
    "core": ["fin", "age", "prio"],
    "full": CFG.covariates,
}

rows = []
for name, covs in model_specs.items():
    m = CoxPHFitter().fit(df[[CFG.duration_col, CFG.event_col, *covs]], CFG.duration_col, CFG.event_col)
    rows.append({
        "model": name,
        "n_covariates": len(covs),
        "concordance": m.concordance_index_,
        "partial_AIC": m.AIC_partial_,
        "log_likelihood": m.log_likelihood_,
    })

cox_model_compare = pd.DataFrame(rows).sort_values("partial_AIC")
display(cox_model_compare.round(4))

Training concordance index = 0.6403


,model,n_covariates,concordance,partial_AIC,log_likelihood
2,core,3,0.6302,"1,327.7141",-660.8570
1,age + prior,2,0.6332,"1,329.0847",-662.5423
3,full,7,0.6403,"1,331.4953",-658.7477
0,treatment only,1,0.5457,"1,348.9242",-673.4621


### AIC reminder

For likelihood-based models,

$$
AIC=2k-2\ell(\hat\theta).
$$

AIC trades fit against parameter count. Lower is preferable **among models fitted to the same response/sample and using comparable likelihood definitions**.

For semiparametric Cox, `lifelines` reports a **partial AIC** based on the partial likelihood. Do not directly mix it with a fully parametric model's ordinary AIC as though they were identical objective functions.

## 13. Proportional-hazards diagnostics

The Cox model assumes

$$
\frac{h(t\mid X_a)}{h(t\mid X_b)}
$$

is constant in $t$ for fixed covariate contrast.

A standard diagnostic uses **Schoenfeld residuals**. At an event time,

$$
r_i^{Sch}=X_i-\bar X(\hat\beta,t_i).
$$

Under PH, these residuals should not exhibit systematic time dependence.

The Grambsch–Therneau-style proportional-hazard test examines association between scaled Schoenfeld residuals and transformed time.

In [19]:
ph_test = proportional_hazard_test(cph, df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], time_transform="rank")
ph_table = ph_test.summary.sort_values("p")
display(ph_table.round(5))

ph_plot_df = ph_table.reset_index().rename(columns={"index": "covariate"})
ph_plot_df["minus_log10_p"] = -np.log10(ph_plot_df["p"].clip(lower=1e-12))

p = figure(
    x_range=ph_plot_df["covariate"].tolist(),
    title="PH diagnostic strength: -log10(p)",
    width=820,
    height=380,
    x_axis_label="Covariate",
    y_axis_label="-log10(p)",
)
p.vbar(x=ph_plot_df["covariate"], top=ph_plot_df["minus_log10_p"], width=0.7, alpha=0.7)
p.add_layout(Span(location=-np.log10(0.05), dimension="width", line_dash="dashed", line_width=2))
p.xaxis.major_label_orientation = 0.8
show(p)

,test_statistic,p,-log2(p)
age,11.4535,0.0007,10.4526
wexp,7.3149,0.0068,7.1921
race,1.4265,0.2323,2.1057
mar,0.7095,0.3996,1.3233
paro,0.1343,0.7140,0.4859
prio,0.0188,0.8908,0.1668
fin,0.0151,0.9023,0.1482


In [20]:
# Visual Schoenfeld residual check for the covariate with the smallest PH-test p-value.
worst_covariate = ph_table.index[0]
schoenfeld = cph.compute_residuals(df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], kind="schoenfeld")

sch = pd.DataFrame({
    "week": df.loc[schoenfeld.index, CFG.duration_col],
    "residual": schoenfeld[worst_covariate],
}).sort_values("week")

# Rolling mean is used only as a visual trend guide, not as an inferential procedure.
sch["rolling_mean"] = sch["residual"].rolling(window=max(7, len(sch)//12), center=True, min_periods=3).mean()

p = figure(
    title=f"Schoenfeld residuals vs time: {worst_covariate}",
    width=820,
    height=400,
    x_axis_label="Event week",
    y_axis_label="Schoenfeld residual",
)
p.scatter(sch["week"], sch["residual"], size=7, alpha=0.55, legend_label="Residual")
p.line(sch["week"], sch["rolling_mean"], line_width=3, legend_label="Rolling trend")
p.add_layout(Span(location=0, dimension="width", line_dash="dashed", line_width=1.5))
p.legend.location = "top_right"
show(p)

### What to do if PH fails

A small diagnostic p-value is not the end of the analysis. Possible responses depend on the scientific question:

1. **Model a time-varying effect** such as $\beta(t)$.
2. **Stratify** on a problematic categorical covariate if its coefficient is not itself the main estimand.
3. Use a different modelling family, e.g. AFT.
4. Add nonlinear transformations if the apparent violation is actually functional-form misspecification.
5. Consider whether the violation is practically important rather than mechanically reacting to a threshold.

## 14. Martingale and deviance residuals

A Cox martingale residual is conceptually

$$
M_i=\delta_i-\hat H_i(T_i).
$$

It compares observed event count ($0$ or $1$ here) with fitted cumulative event intensity.

Martingale residuals are asymmetric but useful for checking covariate functional form. Deviance residuals transform martingales into a more symmetric diagnostic useful for unusual observations.

In [21]:
mart = cph.compute_residuals(df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], kind="martingale")
dev = cph.compute_residuals(df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], kind="deviance")

resid_df = pd.DataFrame({
    "age": df.loc[mart.index, "age"],
    "prio": df.loc[mart.index, "prio"],
    "martingale": mart["martingale"],
    "deviance": dev.loc[mart.index, "deviance"],
})

display(resid_df.describe().T)

p1 = PlotFactory.scatter(resid_df["age"], resid_df["martingale"], "Martingale residual vs age", "Age", "Martingale residual")
p2 = PlotFactory.scatter(resid_df["prio"], resid_df["deviance"], "Deviance residual vs prior count", "Prior count", "Deviance residual")
show(gridplot([[p1], [p2]]))

,count,mean,std,min,25%,50%,75%,max
age,432.0000,24.5972,6.1134,17.0000,20.0000,23.0000,27.0000,44.0000
prio,432.0000,2.9838,2.8961,0.0000,1.0000,2.0000,4.0000,18.0000
martingale,432.0000,0.0000,0.5131,-1.0939,-0.3307,-0.2041,0.5374,0.9986
deviance,432.0000,-0.1481,1.0409,-1.4791,-0.8132,-0.6389,0.6851,3.3353


### Residual-reading heuristic

For martingale residual versus a continuous covariate:

- random cloud around a smooth level: linear effect may be adequate;
- curvature: consider spline/polynomial/transformation;
- sharp isolated points: investigate influence/data quality;
- fan shape or structured pattern: model may be missing interactions or heterogeneity.

Residuals diagnose **model structure**, not merely “bad rows”.

## 15. Stratified Cox model

A stratified Cox model allows separate baseline hazards:

$$
h_k(t\mid X)=h_{0k}(t)e^{X^T\beta}.
$$

The covariate effects $\beta$ are shared, but each stratum has its own $h_{0k}(t)$.

This is useful when a categorical variable strongly changes baseline risk and/or violates PH, while its own coefficient is not the parameter of interest.

Below we demonstrate stratifying by `wexp`. The statistical lesson matters more than whether this is the optimal substantive model for the historical dataset.

In [22]:
strat_covs = [c for c in CFG.covariates if c != "wexp"]
cph_strat = CoxPHFitter().fit(
    df[[CFG.duration_col, CFG.event_col, "wexp", *strat_covs]],
    CFG.duration_col,
    CFG.event_col,
    strata=["wexp"],
)

comparison = pd.DataFrame({
    "model": ["ordinary Cox", "stratified on wexp"],
    "concordance": [cph.concordance_index_, cph_strat.concordance_index_],
    "partial_AIC": [cph.AIC_partial_, cph_strat.AIC_partial_],
})
display(comparison.round(4))

display(cph_strat.summary[["coef", "exp(coef)", "se(coef)", "p"]].round(4))

,model,concordance,partial_AIC
0,ordinary Cox,0.6403,"1,331.4953"
1,stratified on wexp,0.6124,"1,173.7715"


,coef,exp(coef),se(coef),p
covariate,,,,
fin,-0.3802,0.6838,0.1913,0.0469
race,0.3066,1.3588,0.3080,0.3196
mar,-0.4539,0.6352,0.3817,0.2345
paro,-0.0827,0.9206,0.1957,0.6724
age,-0.0582,0.9434,0.0221,0.0083
prio,0.0907,1.0950,0.0287,0.0016


### Trade-off of stratification

You gain flexibility in baseline hazard but lose a direct single coefficient for the stratifying variable.

So stratification answers:

> “Can I control for different baseline risk shapes while estimating the effects of the other variables?”

It does **not** answer:

> “What is one hazard ratio for the stratifying variable?”

## 16. Time-varying Cox effects

A fixed Cox effect assumes

$$
h(t\mid X)=h_0(t)e^{\beta X}.
$$

A time-dependent formulation allows

$$
h_i(t)=h_0(t)e^{\beta_1 X_i+\beta_2X_i g(t)}.
$$

Then the effective coefficient is

$$
\beta(t)=\beta_1+\beta_2g(t).
$$

To fit such models, each subject is represented in **start-stop intervals** so covariates can vary over time.

Below we use episodic expansion and interact `age` with time as a pedagogical demonstration. This is a model experiment, not a claim that this interaction is substantively correct.

In [23]:
# Build a long/start-stop representation in 4-week episodes.
long_df = to_episodic_format(
    df[[CFG.duration_col, CFG.event_col, "fin", "age", "prio"]].copy(),
    duration_col=CFG.duration_col,
    event_col=CFG.event_col,
    time_gaps=4.0,
)

# Scale age to make the interaction numerically well behaved.
long_df["age_centered"] = long_df["age"] - df["age"].mean()
long_df["age_x_log_time"] = long_df["age_centered"] * np.log1p(long_df["stop"])

ctv = CoxTimeVaryingFitter(penalizer=0.01)
ctv.fit(
    long_df[["id", "start", "stop", CFG.event_col, "fin", "prio", "age_centered", "age_x_log_time"]],
    id_col="id",
    start_col="start",
    stop_col="stop",
    event_col=CFG.event_col,
)

display(ctv.summary[["coef", "exp(coef)", "se(coef)", "p"]].round(4))
print(f"Long-format rows: {len(long_df):,} from original {len(df):,} subjects")

,coef,exp(coef),se(coef),p
covariate,,,,
fin,-0.2410,0.7858,0.1571,0.1248
prio,0.0759,1.0788,0.0251,0.0025
age_centered,-0.0203,0.9799,0.0183,0.2690
age_x_log_time,-0.0100,0.9900,0.0056,0.0721


Long-format rows: 4,991 from original 432 subjects


### How to interpret the age-time interaction

With

$$
\eta(t)=\beta_1(age-\bar age)+\beta_2(age-\bar age)\log(1+t),
$$

the age effect at time $t$ is

$$
\beta_{age}(t)=\beta_1+\beta_2\log(1+t).
$$

Thus

$$
HR_{+1\ age}(t)=e^{\beta_{age}(t)}.
$$

A nonzero interaction means one constant hazard ratio is insufficient to describe the covariate effect across follow-up.

## 17. Accelerated Failure Time (AFT) models

Cox models hazard multiplicatively. AFT models time directly:

$$
\log T=X^T\beta+\sigma\varepsilon.
$$

Exponentiating gives

$$
T=e^{X^T\beta}e^{\sigma\varepsilon}.
$$

Therefore $e^{\beta_j}$ has a **time-scale interpretation** rather than a hazard-ratio interpretation.

For example, under a model parameterization where the coefficient enters the log time scale, $e^{\beta_j}=1.20$ indicates an approximately 20% multiplicative expansion of the characteristic event-time scale for a one-unit covariate increase, all else fixed.

Different choices for $\varepsilon$ produce Weibull, log-normal and log-logistic AFT models.

In [24]:
aft_models = {
    "Weibull AFT": WeibullAFTFitter(),
    "Log-normal AFT": LogNormalAFTFitter(),
    "Log-logistic AFT": LogLogisticAFTFitter(),
}

aft_rows = []
fitted_aft = {}
model_df = df[[CFG.duration_col, CFG.event_col, *CFG.covariates]].copy()

for name, model in aft_models.items():
    model.fit(model_df, duration_col=CFG.duration_col, event_col=CFG.event_col)
    fitted_aft[name] = model
    aft_rows.append({
        "model": name,
        "AIC": model.AIC_,
        "log_likelihood": model.log_likelihood_,
        "concordance": model.concordance_index_,
    })

aft_compare = pd.DataFrame(aft_rows).sort_values("AIC")
display(aft_compare.round(4))

,model,AIC,log_likelihood,concordance
0,Weibull AFT,"1,377.8331",-679.9166,0.6401
2,Log-logistic AFT,"1,377.8770",-679.9385,0.6438
1,Log-normal AFT,"1,384.4693",-683.2346,0.6452


In [25]:
# Compare predicted survival for the same representative median-risk profile.
profile = profiles.loc[["50th pct risk"]]

p = figure(
    title="AFT model survival predictions for one representative profile",
    width=820,
    height=430,
    x_axis_label="Week",
    y_axis_label="Predicted survival",
)

for name, model in fitted_aft.items():
    sf = model.predict_survival_function(profile)
    p.line(sf.index, sf.iloc[:, 0], line_width=2.5, legend_label=name)

cox_sf = cph.predict_survival_function(profile)
p.line(cox_sf.index, cox_sf.iloc[:, 0], line_width=2.5, line_dash="dashed", legend_label="Cox PH")
p.legend.location = "bottom_left"
p.legend.click_policy = "hide"
show(p)

### PH versus AFT: different scientific questions

Cox PH asks:

> How does a covariate multiply the instantaneous event hazard?

AFT asks:

> How does a covariate stretch or compress the event-time scale?

Neither is universally superior. The correct model depends on which structural assumption better approximates the data and which effect scale is scientifically meaningful.

## 18. Penalized Cox regression

With many or correlated covariates, maximize a penalized objective such as

$$
\ell_p(\beta)
=
\ell(\beta)
-
\lambda\left[
\frac{1-\alpha}{2}\|\beta\|_2^2
+
\alpha\|\beta\|_1
\right].
$$

- $\alpha=0$: ridge-like shrinkage.
- $\alpha=1$: lasso-like sparsity.
- intermediate $\alpha$: elastic-net behavior.

Penalization trades variance reduction against bias and is especially important when $p$ is not small relative to the number of observed events.

In [33]:
penalty_grid = [0.0, 0.01, 0.1, 0.5, 1.0]
rows = []
coef_paths = []

for penalty in penalty_grid:
    model = CoxPHFitter(penalizer=penalty, l1_ratio=0.0)
    model.fit(df[[CFG.duration_col, CFG.event_col, *CFG.covariates]], CFG.duration_col, CFG.event_col)
    rows.append({
        "penalizer": penalty,
        "concordance": model.concordance_index_,
        "partial_AIC": model.AIC_partial_,
        "coef_l2_norm": float(np.linalg.norm(model.params_.values)),
    })
    for cov, beta in model.params_.items():
        coef_paths.append({"penalizer": penalty, "covariate": cov, "beta": beta})

penalty_results = pd.DataFrame(rows)
display(penalty_results.round(4))

coef_paths = pd.DataFrame(coef_paths)
p = figure(
    title="Ridge penalization shrinks Cox coefficients",
    width=820,
    height=430,
    x_axis_label="Penalizer",
    y_axis_label="Coefficient",
    x_axis_type="linear",
)
for cov, g in coef_paths.groupby("covariate"):
    p.line(g["penalizer"], g["beta"], line_width=2, legend_label=cov)
    p.scatter(g["penalizer"], g["beta"], size=6)
p.legend.location = "top_right"
p.legend.click_policy = "hide"
show(p)

,penalizer,concordance,partial_AIC,coef_l2_norm
0,0.0000,0.6403,"1,331.4953",0.6870
1,0.0100,0.6401,"1,332.5982",0.6626
2,0.1000,0.6453,"1,339.5662",0.5143
3,0.5000,0.6447,"1,351.4469",0.2750
4,1.0000,0.6448,"1,356.3287",0.1766


### What this experiment teaches

As $\lambda$ increases:

- $\|\hat\beta\|$ generally shrinks;
- variance tends to fall;
- bias rises;
- training discrimination may stay similar or decline;
- out-of-sample stability can improve.

For a real predictive project, choose penalty strength using **cross-validation**, not training fit alone.

## 19. Bayesian survival: conjugate exponential example on the same data

This section uses an intentionally simple constant-hazard model to expose Bayesian mechanics.

Assume

$$
T_i\mid\lambda\sim\text{Exponential}(\lambda).
$$

With right censoring,

$$
L(\lambda)\propto
\lambda^D
\exp\left(-\lambda\sum_i t_i\right).
$$

Take a Gamma prior in shape-rate form:

$$
\lambda\sim\mathrm{Gamma}(a,b).
$$

Then conjugacy gives

$$
\boxed{
\lambda\mid D
\sim
\mathrm{Gamma}
\left(a+D,\;b+\sum_i t_i\right)
}.
$$

The posterior predictive survival probability integrates over parameter uncertainty:

$$
P(T_{new}>t\mid D)
=E[e^{-\lambda t}\mid D].
$$

For a Gamma posterior this has closed form:

$$
\boxed{
S_{pred}(t)
=
\left(
\frac{b'}{b'+t}
\right)^{a'}
}
$$

where $a'=a+D$ and $b'=b+\sum_i t_i$.

In [32]:
# Weakly informative Gamma(shape, rate) prior on the weekly hazard.
a0 = 1.0
b0 = 100.0

D = float(df[CFG.event_col].sum())
exposure = float(df[CFG.duration_col].sum())

a_post = a0 + D
b_post = b0 + exposure

posterior_mean_rate = a_post / b_post
posterior_median_rate = stats.gamma.median(a_post, scale=1/b_post)
posterior_ci = stats.gamma.ppf([0.025, 0.975], a_post, scale=1/b_post)

bayes_summary = pd.DataFrame({
    "quantity": ["posterior mean hazard", "posterior median hazard", "2.5%", "97.5%"],
    "value": [posterior_mean_rate, posterior_median_rate, posterior_ci[0], posterior_ci[1]],
})
display(bayes_summary)

,quantity,value
0,posterior mean hazard,0.0058
1,posterior median hazard,0.0058
2,2.5%,0.0048
3,97.5%,0.0069


In [31]:
t = np.linspace(0, df[CFG.duration_col].max(), 200)
posterior_predictive_survival = (b_post / (b_post + t)) ** a_post
plug_in_survival = np.exp(-posterior_mean_rate * t)
km_values = km.predict(t).values

show(PlotFactory.line(
    t,
    {
        "KM": km_values,
        "Bayesian posterior predictive (Exponential-Gamma)": posterior_predictive_survival,
        "Plug-in exponential at posterior mean": plug_in_survival,
    },
    "Bayesian predictive survival vs nonparametric KM",
    "Week",
    "Survival probability",
))

### Why posterior predictive and plug-in curves differ

The plug-in curve uses one rate:

$$
e^{-E[\lambda\mid D]t}.
$$

The posterior predictive curve computes

$$
E[e^{-\lambda t}\mid D].
$$

In general,

$$
E[g(\lambda)]\ne g(E[\lambda]).
$$

The predictive distribution integrates parameter uncertainty rather than pretending $\lambda$ is known exactly.

The much bigger modelling question remains whether a **constant hazard** is credible. Bayesian inference does not rescue a badly chosen likelihood family.

## 20. Optional competing-risks extension

The Rossi endpoint has one recorded event type (`arrest`), so it cannot identify competing causes of failure. To explain the ST5212 competing-risk machinery without inventing meanings for the real data, this section creates an **explicitly synthetic two-cause extension**.

For cause $k$, the cause-specific hazard is

$$
h_k(t)=\lim_{\Delta t\to0}
\frac{P(t\le T<t+\Delta t,J=k\mid T\ge t)}{\Delta t}.
$$

Overall hazard is

$$
h(t)=\sum_k h_k(t).
$$

The cumulative incidence function is

$$
\boxed{
F_k(t)=P(T\le t,J=k)
=
\int_0^t S(u^-)h_k(u)\,du
}.
$$

The presence of overall survival $S(u^-)$ is crucial: a subject must remain free of **all competing events** until cause $k$ can occur.

In [30]:
# Pedagogical synthetic competing-risk data; do not interpret causes substantively.
rng = np.random.default_rng(RANDOM_STATE)
n = 600
cause1 = rng.exponential(scale=18, size=n)
cause2 = rng.exponential(scale=30, size=n)
censor = rng.uniform(20, 52, size=n)

observed_time = np.minimum.reduce([cause1, cause2, censor])
event_type = np.where(
    observed_time == censor,
    0,
    np.where(cause1 <= cause2, 1, 2)
)

cr = pd.DataFrame({"time": observed_time, "event_type": event_type})
display(cr["event_type"].value_counts().sort_index().rename(index={0:"censored",1:"cause 1",2:"cause 2"}))

aj1 = AalenJohansenFitter().fit(cr["time"], cr["event_type"], event_of_interest=1)
aj2 = AalenJohansenFitter().fit(cr["time"], cr["event_type"], event_of_interest=2)

cif1 = aj1.cumulative_density_.reset_index()
cif2 = aj2.cumulative_density_.reset_index()

p = figure(
    title="Aalen–Johansen cumulative incidence functions (synthetic extension)",
    width=820,
    height=420,
    x_axis_label="Time",
    y_axis_label="Cumulative incidence",
)
p.step(cif1.iloc[:, 0], cif1.iloc[:, 1], mode="after", line_width=3, legend_label="Cause 1 CIF")
p.step(cif2.iloc[:, 0], cif2.iloc[:, 1], mode="after", line_width=3, legend_label="Cause 2 CIF")
p.legend.location = "top_left"
show(p)

event_type
censored     28
cause 1     350
cause 2     222
Name: count, dtype: int64

## 21. Left truncation / delayed entry: conceptual extension

Censoring means a subject is observed but the event time is only partially known. **Truncation** changes who enters the sample.

If a subject becomes observable only after time $L_i$, then inference conditions on

$$
T_i>L_i.
$$

For an exact failure at $t_i>L_i$, the conditional likelihood contribution is

$$
\frac{f(t_i)}{S(L_i)}.
$$

Algorithmically, the most important consequence is the risk set:

$$
i\in R(t)
\quad\text{only if}\quad
L_i<t\le Y_i.
$$

So delayed entry is largely a **risk-set construction problem**. `lifelines.KaplanMeierFitter.fit(..., entry=...)` and Cox start-stop representations can handle it when true entry times are available.

We deliberately do not fabricate delayed-entry times for the Rossi data.

## 22. End-to-end model selection checklist

A defensible survival workflow is not “fit Cox and report p-values”. Use this sequence.

### Step 1 — Define the estimand

Are you interested in:

- $S(t)$?
- median event time?
- hazard ratio?
- time ratio?
- cause-specific incidence?
- individualized risk ranking?

Different questions imply different models.

### Step 2 — Understand observation

Check censoring, truncation, competing events and time origin.

### Step 3 — Start nonparametrically

Use Kaplan–Meier and Nelson–Aalen to understand shape before imposing regression structure.

### Step 4 — Compare important groups

Use survival curves and log-rank-type tests, while remembering these are unadjusted comparisons.

### Step 5 — Fit regression

Choose among Cox PH, stratified/time-varying Cox, or parametric/AFT models based on assumptions and effect interpretation.

### Step 6 — Diagnose

Check PH, functional form, residuals, influential observations and data support in the tail.

### Step 7 — Evaluate

Use discrimination such as C-index, calibration at relevant horizons, and ideally out-of-sample validation.

### Step 8 — Interpret on the right scale

Hazard ratio $\ne$ probability ratio $\ne$ survival-time ratio.

## 23. Summary table: what each ST5212 method is doing

| Method | Core object | Main computation | Key assumption / limitation |
|---|---|---|---|
| Kaplan–Meier | $S(t)$ | $\prod(1-d_j/n_j)$ | independent/non-informative censoring |
| Nelson–Aalen | $H(t)$ | $\sum d_j/n_j$ | same censoring logic |
| Log-rank | group comparison | $\sum(O-E)$ | most natural under PH-like group separation |
| Exponential | $S,h$ | censored MLE | constant hazard |
| Weibull | $S,h$ | censored MLE | monotone hazard shape |
| Cox PH | hazard ratio | partial likelihood over risk sets | proportional hazards |
| Stratified Cox | hazard ratio | separate baseline hazards | no HR for stratifier |
| Time-varying Cox | $\beta(t)$ or $X(t)$ | start-stop risk sets | more complex specification |
| AFT | event-time scale | parametric censored likelihood | distributional assumption |
| Penalized Cox | sparse/shrunk $\beta$ | penalized partial likelihood | tuning required |
| Aalen–Johansen | competing-risk CIF | multistate product-integral logic | competing-event structure must be defined |
| Bayesian survival | posterior/predictive | likelihood × prior | depends on both likelihood and prior |

## 24. Exercises

### Exercise 1 — Derive a Weibull median

Given

$$
S(t)=\exp[-(t/\lambda)^\rho],
$$

solve $S(m)=0.5$ for $m$.

### Exercise 2 — KM by another binary covariate

Repeat the KM + log-rank workflow for `wexp` or `paro`. Explain why a group comparison is not a causal interpretation.

### Exercise 3 — Nonlinear age effect

Fit a Cox model using $age^2$ or a spline representation. Compare partial AIC and residual patterns against the linear-age model.

### Exercise 4 — PH sensitivity

Take the variable with the smallest PH-test p-value. Compare:

1. ordinary Cox,
2. stratified Cox if categorical,
3. a time interaction.

Explain how the estimand changes.

### Exercise 5 — Penalization

Extend the ridge grid with elastic-net penalties and inspect coefficient paths.

### Exercise 6 — Prediction

Choose two real rows with substantially different Cox partial hazards and compare their predicted $S(t\mid X)$ at weeks 10, 26 and 52.

### Exercise 7 — Competing risks

Change the synthetic cause-2 rate and explain why the cause-1 cumulative incidence changes even when the cause-1 hazard mechanism itself is unchanged.

This last exercise is particularly important: **competing events alter cumulative incidence through the probability of remaining event-free.**

## 25. Suggested extensions for a second notebook

A natural follow-up can move from classical ST5212 into modern survival ML:

- Random Survival Forests;
- Gradient Boosting Survival Analysis;
- Coxnet (Lasso/Elastic Net);
- DeepSurv / neural Cox models;
- time-dependent Brier score;
- IPCW evaluation;
- calibration curves at fixed horizons;
- nested cross-validation;
- competing-risk prediction;
- recurrent-event models;
- frailty/random-effects survival models;
- spline-based flexible parametric survival models.

The classical theory in this notebook remains foundational because these methods still need censoring-aware losses, risk sets, survival functions and proper time-to-event evaluation.

## References / documentation used for the case study

- NUS Department of Statistics & Data Science — MSc Statistics curriculum / ST5212 listing.
- `lifelines` datasets documentation — Rossi dataset description.
- `lifelines` survival regression documentation — Cox PH, penalization and prediction.
- `lifelines` proportional-hazards assumption tutorial — Schoenfeld-based diagnostics on Rossi.
- Cox, D. R. (1972), *Regression Models and Life-Tables*.
- Kaplan, E. L. & Meier, P. (1958), *Nonparametric Estimation from Incomplete Observations*.

Useful online documentation:

- https://lifelines.readthedocs.io/en/stable/lifelines.datasets.html
- https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html
- https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html